# OCR Ingest & RAG with Visual Grounding

This notebook covers two phases:

| Phase | What happens |
|---|---|
| **1. Ingest** | Parse a PDF with Landing AI ADE → embed chunks → store in Postgres with bounding-box metadata |
| **2. RAG + Visual Grounding** | Query the vector store → retrieve matching chunks → highlight their bounding boxes on the original PDF page |

The bbox columns (`bbox_left`, `bbox_top`, `bbox_right`, `bbox_bottom`) are stored in Postgres as normalised coordinates (0–1 range) so they can be applied to any render resolution.

## 1. Imports & Configuration

In [ ]:
import os
import sys
from pathlib import Path

import pymupdf
from dotenv import load_dotenv
from IPython.display import Image as DisplayImage
from IPython.display import Markdown, display
from PIL import Image as PILImage
from PIL import ImageDraw

# Make src/ importable from the notebook
PROJECT_ROOT = Path("../").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(PROJECT_ROOT / ".env", override=True)

# Project imports (after sys.path is set)
from config import EMBEDDING_MODEL, PG_TABLE
from helper import CHUNK_TYPE_COLORS, extract_chunk_image, print_document
from ocr_extraction import DocumentAI
from vector_store import get_retriever, get_store

print("Imports OK")

In [ ]:
# ── Configure these paths before running ────────────────────────────────────
CASE_NUMBER   = "SteveGoodman"   # must match the sub-directory under data/
DOCUMENT_PATH = PROJECT_ROOT / f"data/{CASE_NUMBER}/5573DraftAccounts.pdf"
# ─────────────────────────────────────────────────────────────────────────────

assert DOCUMENT_PATH.exists(), f"Document not found: {DOCUMENT_PATH}"
print(f"Document : {DOCUMENT_PATH}")
print(f"Case     : {CASE_NUMBER}")

---
## 2. Phase 1 — Ingest

### 2a. Parse with Landing AI ADE
Calls `client.parse()` which returns:
- `.splits[]` — per-page content
- `.chunks[]` — individual content blocks (text / table / figure …)
- `.grounding` — `dict[chunk_id, Grounding]` with page + normalised bbox

In [ ]:
document = DocumentAI(DOCUMENT_PATH, case_number=CASE_NUMBER)
document.parse()

parse_result = document.parse_result
print(f"Pages  : {len(parse_result.splits)}")
print(f"Chunks : {len(parse_result.chunks)}")
print(f"Grounding entries: {len(parse_result.grounding)}")

### 2b. Inspect a grounding entry
Each entry in `parse_result.grounding` is keyed by `chunk_id` and has:
- `.page` — 0-indexed page number
- `.type` — chunk type string (`chunkText`, `chunkTable`, …)
- `.box.left / .top / .right / .bottom` — normalised (0–1) coordinates

In [ ]:
sample_id, sample_g = next(iter(parse_result.grounding.items()))
print(f"chunk_id : {sample_id}")
print(f"page     : {sample_g.page}")
print(f"type     : {sample_g.type}")
print(f"box      : left={sample_g.box.left:.4f}  top={sample_g.box.top:.4f}  "
      f"right={sample_g.box.right:.4f}  bottom={sample_g.box.bottom:.4f}")

### 2c. Visualise all grounded chunks (optional sanity check)

In [ ]:
TARGET_PAGE = 0   # change to see other pages

pdf = pymupdf.open(DOCUMENT_PATH)
page = pdf[TARGET_PAGE]
pix  = page.get_pixmap(matrix=pymupdf.Matrix(2, 2))
img  = PILImage.frombytes("RGB", [pix.width, pix.height], pix.samples)
pdf.close()

draw = ImageDraw.Draw(img)
img_w, img_h = img.size
for cid, g in parse_result.grounding.items():
    if g.page != TARGET_PAGE:
        continue
    b = g.box
    color = CHUNK_TYPE_COLORS.get(g.type, (128, 128, 128))
    x1, y1 = int(b.left * img_w), int(b.top * img_h)
    x2, y2 = int(b.right * img_w), int(b.bottom * img_h)
    draw.rectangle([x1, y1, x2, y2], outline=color, width=2)

display(img.resize((int(img_w * 0.5), int(img_h * 0.5))))

In [ ]:
# Clear existing rows for this case before re-ingesting
from sqlalchemy import text
from sqlalchemy.ext.asyncio import create_async_engine

engine = create_async_engine(
    f"postgresql+asyncpg://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)
async with engine.begin() as conn:
    result = await conn.execute(
        text(f'DELETE FROM "{PG_TABLE}" WHERE case_number = :case'),
        {"case": CASE_NUMBER},
    )
    print(f"Deleted {result.rowcount} rows for case '{CASE_NUMBER}'")
await engine.dispose()

### 2d. Embed chunks and store in Postgres

`embed_and_store()` persists **bounding-box coordinates** alongside each chunk so they can be retrieved for visual grounding later.

> **First run only** — run `uv run python src/vector_store.py` from the project root to create / migrate the table schema before embedding.

In [ ]:
stored = await document.embed_and_store()
print(f"Stored {stored} chunks in table '{PG_TABLE}'.")

---
## 3. Phase 2 — RAG with Visual Grounding

### 3a. Connect to the vector store

In [ ]:
store = await get_store()
print("Connected to PGVectorStore")

### 3b. Visual-grounding query function

For each retrieved chunk the function:
1. Renders the source PDF page at 2× resolution
2. Crops to the stored bounding box (with padding)
3. Displays the cropped image alongside the text excerpt

In [ ]:
async def rag_query(
    question: str,
    k: int = 5,
    chunk_type: str | None = None,
    show_images: bool = True,
):
    """Semantic search against the Postgres vector store with visual grounding.

    Args:
        question:    Natural-language query.
        k:           Max chunks to retrieve.
        chunk_type:  Optional filter — e.g. 'chunkTable', 'chunkText'.
        show_images: Whether to render cropped bounding-box images.

    Returns:
        List of retrieved LangChain Documents.
    """
    retriever = get_retriever(store, CASE_NUMBER, k=k, chunk_type=chunk_type)
    docs = await retriever.ainvoke(question)

    display(Markdown(f"### Query: *{question}*\n---"))
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        page_num    = meta.get("page_num") or 0
        ctype       = meta.get("chunk_type") or "unknown"
        chunk_id    = meta.get("chunk_id") or ""
        bbox_left   = meta.get("bbox_left")
        bbox_top    = meta.get("bbox_top")
        bbox_right  = meta.get("bbox_right")
        bbox_bottom = meta.get("bbox_bottom")

        display(Markdown(
            f"**Result {i}** — type: `{ctype}` | page: {page_num} | chunk: `{chunk_id[:8] if chunk_id else 'N/A'}…`\n\n"
            f"```\n{doc.page_content[:300]}\n```"
        ))

        if show_images and all(v is not None for v in [bbox_left, bbox_top, bbox_right, bbox_bottom]):
            bbox = [bbox_left, bbox_top, bbox_right, bbox_bottom]
            img_bytes = extract_chunk_image(
                pdf_path=DOCUMENT_PATH,
                page_num=page_num,
                bbox=bbox,
                highlight=True,
                padding=15,
            )
            if img_bytes:
                display(DisplayImage(data=img_bytes))
            else:
                print("  (could not extract chunk image)")
        elif show_images:
            print(f"  (no bbox stored — re-ingest this document to get visual grounding)")
        print("-" * 60)

    return docs

print("rag_query() defined")

### 3c. Run a query

In [ ]:
results = await rag_query(
    "What is the turnover and profit for the company?",
    k=4,
)

### 3d. Hybrid search — tables only

Narrow retrieval to `chunkTable` chunks to focus on financial figures.

In [ ]:
results_tables = await rag_query(
    "balance sheet fixed assets and debtors",
    k=3,
    chunk_type="chunkTable",
)

---
## 4. Full RAG Chain with Claude

Combine the PGVectorStore retriever with a LangChain RAG chain backed by Claude.

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

system_prompt = (
    "You are a financial analyst assistant. "
    "Use ONLY the retrieved document chunks below to answer. "
    "If the answer is not in the context, say so.\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

retriever = get_retriever(store, CASE_NUMBER, k=6)
doc_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, doc_chain)

print("RAG chain ready")

In [ ]:
response = await rag_chain.ainvoke({"input": "What are the company's total turnover and net profit for the most recent year?"})
display(Markdown(f"**A:** {response['answer']}"))

### 4a. Ground the answer — show source chunks

In [ ]:
display(Markdown("**Source chunks used to generate the answer:**"))
for doc in response["context"]:
    meta = doc.metadata
    page_num    = meta.get("page_num", 0)
    ctype       = meta.get("chunk_type", "unknown")
    bbox_left   = meta.get("bbox_left")
    bbox_top    = meta.get("bbox_top")
    bbox_right  = meta.get("bbox_right")
    bbox_bottom = meta.get("bbox_bottom")

    display(Markdown(f"*page {page_num} | {ctype}*\n```\n{doc.page_content[:200]}\n```"))

    if all(v is not None for v in [bbox_left, bbox_top, bbox_right, bbox_bottom]):
        img_bytes = extract_chunk_image(
            pdf_path=DOCUMENT_PATH,
            page_num=page_num,
            bbox=[bbox_left, bbox_top, bbox_right, bbox_bottom],
            highlight=True,
            padding=15,
        )
        if img_bytes:
            display(DisplayImage(data=img_bytes))